# Cross-Dataset Evaluation: RAF-DB Models → CK+

Đánh giá khả năng generalization của 3 model (CNN, DAN, POSTER) đã train trên RAF-DB lên dataset CK+.

## 1. Import & Config

In [ ]:
import os, sys, warnings, collections, gc
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
CKP_PATH = 'data/ckplus'
SAVE_DIR = 'outputs/evaluation/ckplus'
os.makedirs(SAVE_DIR, exist_ok=True)

# CK+ classes và mapping sang RAF-DB labels (0-indexed)
CK_CLASSES = ['surprise', 'fear', 'disgust', 'happy', 'sadness', 'anger']
# skip contempt (54 ảnh) + không có class Neutral tương ứng
CK_TO_RAF = {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
NUM_CLASSES = 6

## 2. Load CK+ dataset

In [ ]:
FOLDER_MAP = {'surprise': 0, 'fear': 1, 'disgust': 2, 'happy': 3, 'sadness': 4, 'anger': 5}

ck_paths, ck_labels = [], []
for folder, label in FOLDER_MAP.items():
    dir_path = os.path.join(CKP_PATH, folder)
    if not os.path.exists(dir_path): continue
    for fname in os.listdir(dir_path):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            ck_paths.append(os.path.join(dir_path, fname))
            ck_labels.append(label)

ck_paths = np.array(ck_paths)
ck_labels = np.array(ck_labels)

print(f'CK+ loaded: {len(ck_paths)} images')
for i, name in enumerate(CK_CLASSES):
    count = (ck_labels == i).sum()
    print(f'  {name:12s}: {count}')

## 3. Dataset class cho từng model

In [ ]:
class CKPlusDataset(Dataset):
    def __init__(self, paths, labels, transform=None, size=(224, 224)):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.size = size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((*self.size, 3), dtype=np.uint8)
        # CK+ là grayscale, convert sang RGB
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 1:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Transform cho PyTorch models (DAN, POSTER) — ImageNet norm
pt_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 4. Đánh giá Baseline CNN (TensorFlow)

In [ ]:
import tensorflow as tf
from utils.models import build_baseline_cnn
from utils.data_loader import RAF_MEAN, RAF_STD

print('Loading CNN model...')
cnn_model = build_baseline_cnn(input_shape=(100, 100, 3), num_classes=7, dropout_rate=0.25)

ckpt_path = 'outputs/models/baseline_cnn.keras'
if os.path.exists(ckpt_path):
    cnn_model = tf.keras.models.load_model(ckpt_path)
    print(f'Loaded: {ckpt_path}')
else:
    print('Checkpoint not found!')

In [ ]:
def preprocess_cnn(img):
    img = cv2.resize(img, (100, 100))
    img = img.astype('float32')
    img[..., 0] = (img[..., 0] - RAF_MEAN[0]) / RAF_STD[0]
    img[..., 1] = (img[..., 1] - RAF_MEAN[1]) / RAF_STD[1]
    img[..., 2] = (img[..., 2] - RAF_MEAN[2]) / RAF_STD[2]
    return img

print('Running CNN on CK+...')
cnn_preds = []
start = time()
for path in ck_paths:
    img = cv2.imread(path)
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    x = preprocess_cnn(img)
    x = np.expand_dims(x, 0)
    pred = cnn_model.predict(x, verbose=0)[0]
    cnn_preds.append(np.argmax(pred))
cnn_preds = np.array(cnn_preds)
elapsed = time() - start
print(f'Done in {elapsed:.1f}s ({len(ck_paths)/elapsed:.1f} img/s)')

In [ ]:
def eval_results(name, preds, labels):
    overall = (preds == labels).sum() / len(labels) * 100
    per_class = []
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'  Overall Accuracy: {overall:.2f}%')
    for i in range(NUM_CLASSES):
        mask = (labels == i)
        acc = (preds[mask] == i).sum() / mask.sum() * 100
        per_class.append(acc)
        print(f'    {CK_CLASSES[i]:12s}: {acc:.2f}%')
    print(f'    {"Mean":12s}: {np.mean(per_class):.2f}%')
    return overall, per_class

cnn_acc, cnn_pc = eval_results('Baseline CNN on CK+', cnn_preds, ck_labels)

## 5. Đánh giá DAN (PyTorch)

In [ ]:
class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super().__init__()
        from torchvision import models
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)

    def forward(self, x):
        x = self.features(x)
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        weighted_features = (x_flat * att_flat).sum(dim=-1)
        final_features = weighted_features.mean(dim=1)
        out = self.fc(final_features)
        return self.bn(out)

print('Loading DAN model...')
dan_model = DAN(num_class=7, num_head=4)
ckpt = torch.load('outputs/models/best_dan_model.pth', map_location='cpu')
if any(k.startswith('module.') for k in ckpt):
    ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
dan_model.load_state_dict(ckpt)
dan_model = dan_model.to(device).eval()
print('DAN loaded!')

In [ ]:
dan_ds = CKPlusDataset(ck_paths, ck_labels, transform=pt_tf)
dan_loader = DataLoader(dan_ds, batch_size=32, shuffle=False, num_workers=0)

print('Running DAN on CK+...')
dan_preds = []
start = time()
with torch.no_grad():
    for imgs, _ in dan_loader:
        out = dan_model(imgs.to(device))
        dan_preds.extend(out.argmax(1).cpu().tolist())
dan_preds = np.array(dan_preds)
elapsed = time() - start
print(f'Done in {elapsed:.1f}s ({len(ck_paths)/elapsed:.1f} img/s)')

In [ ]:
dan_acc, dan_pc = eval_results('DAN on CK+', dan_preds, ck_labels)

## 6. Đánh giá POSTER

In [ ]:
sys.path.insert(0, os.getcwd())
from models.ir50 import Backbone
from models.mobilefacenet import MobileFaceNet
from models.hyp_crossvit import HyVisionTransformer

class SE_block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d,d), nn.ReLU(), nn.Linear(d,d), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)

class POSTER(nn.Module):
    def __init__(self, num_classes=7, depth=8):
        super().__init__()
        self.face_landback = MobileFaceNet([112,112], 136)
        self.ir_back = Backbone(50, 0.0, 'ir')
        self.ir_layer = nn.Linear(1024, 512)
        self.pyramid_fuse = HyVisionTransformer(
            in_chans=49, q_chanel=49, embed_dim=512,
            depth=depth, num_heads=8, mlp_ratio=2.,
            drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1)
        self.se_block = SE_block(512)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x_face = F.interpolate(x, size=112)
        _, x_face = self.face_landback(x_face)
        x_face = x_face.view(B, -1, 49).transpose(1, 2)
        x_ir = self.ir_layer(self.ir_back(x))
        y = self.se_block(self.pyramid_fuse(x_ir, x_face))
        return self.head(self.dropout(y)), y

print('Loading POSTER model...')
poster_model = POSTER(num_classes=7, depth=8)
ckpt = torch.load('checkpoints/poster_best.pth', map_location='cpu')
sd = ckpt.get('state_dict', ckpt)
try:
    poster_model.load_state_dict(sd)
except:
    # Try mapping body1/2/3
    md = poster_model.state_dict()
    new = collections.OrderedDict()
    key_map = {}
    idx = 0
    for group, count in [('body1', 3), ('body2', 4), ('body3', 14)]:
        for i in range(count):
            key_map[f'{group}.{i}.'] = f'body.{idx}.'
            idx += 1
    for k, v in sd.items():
        k_clean = k.replace('module.', '')
        mapped_k = k_clean
        for old_prefix, new_prefix in key_map.items():
            if k_clean.startswith(old_prefix):
                mapped_k = k_clean.replace(old_prefix, new_prefix)
                break
        if mapped_k in md and md[mapped_k].size() == v.size():
            new[mapped_k] = v
    md.update(new)
    poster_model.load_state_dict(md)
    print(f'  Mapped {len(new)}/{len(md)} layers')

poster_model = poster_model.to(device).eval()
print('POSTER loaded!')

In [ ]:
poster_ds = CKPlusDataset(ck_paths, ck_labels, transform=pt_tf)
poster_loader = DataLoader(poster_ds, batch_size=8, shuffle=False, num_workers=0)

print('Running POSTER on CK+...')
poster_preds = []
start = time()
with torch.no_grad():
    for imgs, _ in poster_loader:
        out, _ = poster_model(imgs.to(device))
        poster_preds.extend(out.argmax(1).cpu().tolist())
poster_preds = np.array(poster_preds)
elapsed = time() - start
print(f'Done in {elapsed:.1f}s ({len(ck_paths)/elapsed:.1f} img/s)')

In [ ]:
poster_acc, poster_pc = eval_results('POSTER on CK+', poster_preds, ck_labels)

## 7. So sánh 3 models

In [ ]:
x = np.arange(NUM_CLASSES)
w = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w, cnn_pc, w, label=f'CNN ({cnn_acc:.1f}%)')
ax.bar(x, dan_pc, w, label=f'DAN ({dan_acc:.1f}%)')
ax.bar(x + w, poster_pc, w, label=f'POSTER ({poster_acc:.1f}%)')
ax.set_xticks(x)
ax.set_xticklabels(CK_CLASSES)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Cross-Dataset Evaluation: RAF-DB Models → CK+')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'model_comparison_ckplus.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Results saved to: {SAVE_DIR}')